In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import pandas as pd
df = pd.read_csv("../data/processed/kidney_clean.csv")
df.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,normal,normal,notpresent,notpresent,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,normal,normal,notpresent,notpresent,...,38.0,6000.0,4.8,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31.0,7500.0,4.8,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [3]:
binary_map = {"yes": 1, "no": 0, "good": 1, "poor": 0, "normal": 1, "abnormal": 0,
              "present": 1, "notpresent": 0, "ckd": 1, "notckd": 0}
cat_cols = ["htn", "dm", "cad", "appet", "pe", "ane", "rbc", "pc", "pcc", "ba", "classification"]
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower().map(binary_map)
df[cat_cols].apply(lambda x: x.unique())

,htn,dm,cad,appet,pe,ane,rbc,pc,pcc,ba,classification
0,1,1,0,1,0,0,1,1,0,0,1
1,0,0,1,0,1,1,0,0,1,1,0


In [4]:
# Domain Feature 1
import numpy as np
df["bun_creatinine_ratio"] = df["bu"] / df["sc"].replace(0, np.nan)

df["bun_creatinine_ratio"] = (
    df["bun_creatinine_ratio"]
      .fillna(df["bun_creatinine_ratio"].median())
)

In [5]:
# Domain Feature 2
df["anemia_ckd_flag"] = (
    (df["ane"] == 1)
    &
    (df["hemo"] < 12)
).astype(int)

In [6]:
# Day 8 leakage fix: StandardScaler removed from this notebook.
# Previously, scaler.fit_transform() ran here on the FULL 400-row dataset,
# before any train/test split existed -- leaking test-set statistics into
# training. kidney_features.csv is now saved with engineered-but-UNSCALED
# numeric features. Scaling happens ONLY inside src/data_loader.py's
# load_and_prepare(), fit on X_train only, AFTER the split.

NUMERIC_COLS = [
    "age","bp","sg","al","su",
    "bgr","bu","sc","sod","pot",
    "hemo","pcv","wc","rc"
]

In [7]:
corr = df[NUMERIC_COLS].corr().abs()
import numpy as np
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.9)]
to_drop

[]

In [8]:
df.to_csv("../data/processed/kidney_features.csv", index=False)
# NOTE: scaler.joblib dump removed -- that scaler was fit on the full
# dataset (leaky). Scaling + the production scaler are now produced
# inside src/data_loader.py's load_and_prepare(), fit on X_train only.

In [9]:
# Self-check disabled pending the matching fix in src/features.py's
# build_features(), which currently still calls scale_numeric() by
# default. Once that function is updated (Day 8 leakage fix, src side),
# re-enable an equivalent check comparing UNSCALED outputs.
# from src.features import build_features
# df_check, scaler_check = build_features(pd.read_csv("../data/processed/kidney_clean.csv"), fit=False, scaler=None)

In [10]:
print(corr.loc["hemo", "pcv"])

0.8474896283048784
